# 01 - RSF benchmark

Random survival forest across the five development folds, at **imputation
m = 1 only**.

**Why one imputation.** RSF is the runtime bottleneck of the whole project -
roughly 15 hours for the full matrix even at reduced settings, against
minutes for every other classical baseline. Between-imputation variance is
negligible at this project's missingness rates (0.26% cold ischemia, ~4.5%
biopsy grades within biopsied rows), so the full m = 1..5 sweep would buy
almost nothing for about five times the cost. **This is a stated deviation in
the paper, not a hidden one** - report it in the methods alongside the
missingness rates that justify it.

**The temporal and exploratory windows are never loaded here.** They are
scored once, at the end of the project, after every modelling decision is
locked. `io_helpers.load_set` raises if asked for them.

Runs unattended for hours: every step is timestamped, and the run is
resumable, so an interruption continues rather than restarting.

In [1]:
# =====================================================================
# CONFIGURATION - the only cell to edit
# =====================================================================
from pathlib import Path

REPO_ROOT    = Path(r"C:\Users\olf3ph\Documents")      # repository root; leave "" to auto-detect
FEATURES_DIR = Path(r"C:\Users\olf3ph\Documents\FEATURES")          # feature-builder output
RESULTS_DIR  = Path(r"C:\Users\olf3ph\Documents\RESULTS")          # benchmark parquet + logs

T_EVAL = [1.0, 3.0, 5.0]          # years
FOLDS  = [0, 1, 2, 3, 4]

# =====================================================================
# Nothing below this line needs editing.
# =====================================================================
import sys, os, json, time
import numpy as np
import pandas as pd

_here = Path.cwd()
if not str(REPO_ROOT):
    # Notebook may be run from notebooks/ or from the repository root.
    REPO_ROOT = _here.parent if (_here / "io_helpers.py").exists() else _here
REPO_ROOT = Path(REPO_ROOT).resolve()

sys.path.insert(0, str(REPO_ROOT / "notebooks"))
import io_helpers as ioh
ioh.add_rp_core_to_path(REPO_ROOT)

FEATURES_DIR = Path(FEATURES_DIR)
RESULTS_DIR = Path(RESULTS_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

assert FEATURES_DIR.exists(), f"FEATURES_DIR does not exist: {FEATURES_DIR}"
ioh.log("configuration", repo=REPO_ROOT, features=FEATURES_DIR,
        results=RESULTS_DIR)

# What the feature builder actually produced. The held_out column is a
# reminder, not a permission: the loader refuses those sets outright.
_avail = ioh.available_sets(FEATURES_DIR)
print(_avail.to_string(index=False))

[2026-08-21 11:51:56] configuration  repo=C:\Users\olf3ph\Documents  features=C:\Users\olf3ph\Documents\FEATURES  results=C:\Users\olf3ph\Documents\RESULTS
     applied_to  imputation  held_out  onehot  embedding
 dev_fold0_test           1     False    True       True
 dev_fold0_test           2     False    True       True
 dev_fold0_test           3     False    True       True
 dev_fold0_test           4     False    True       True
 dev_fold0_test           5     False    True       True
dev_fold0_train           1     False    True       True
dev_fold0_train           2     False    True       True
dev_fold0_train           3     False    True       True
dev_fold0_train           4     False    True       True
dev_fold0_train           5     False    True       True
 dev_fold1_test           1     False    True       True
 dev_fold1_test           2     False    True       True
 dev_fold1_test           3     False    True       True
 dev_fold1_test           4     False    True 

## 1. Single-fit timing probe

**Run this before launching anything.** One fold, timed, with the projected
total printed. A cell that says "projected 40 hours" is worth far more before
the run than after it.

Settings, and why:

| Option | Value | Reason |
|---|---|---|
| `ntree` | 500 | |
| `nodesize` | 30 | larger than the default 15; shallower trees, much cheaper |
| `ntime` | 100 | **not a tuning knob.** Left unset, `rfsrc` evaluates CIFs at every distinct event time - at n = 37,000 that is a multi-gigabyte array per cause |
| `nsplit` | 10 | random split points instead of an exhaustive scan; most of the saving on continuous covariates |
| `perf.type` | `"none"` | switches off OOB error bookkeeping, which nothing here reads - scoring is on held-out folds via `metrics.evaluate` |
| `block.size` | `NULL` | same reason |
| `rf.cores` | all cores | |

In [2]:
from benchmark import BenchmarkCell, coherence_table, run_benchmark

RSF_OPTIONS = dict(ntree=500, nodesize=30, ntime=100, nsplit=10,
                   perf_type="none", block_size=None, rf_cores=None)

# --- FALLBACK, if the probe below exceeds ~30 minutes ------------------
# Drop to ntree=300, but ONLY after confirming the OOB error has plateaued -
# fewer trees is a weaker comparator, and "it was slow" is not a method.
# To check, fit once with perf_type="oob" and inspect the error curve, then
# set the tree count where it flattens.
#     RSF_OPTIONS.update(ntree=300)
# Whatever is used, REPORT IT: an RSF at 300 trees is not an RSF at 1000.
# -----------------------------------------------------------------------

IMPUTATION = 1

train0, test0 = ioh.load_fold(FEATURES_DIR, FOLDS[0], IMPUTATION, "onehot")
print(ioh.describe(train0))
print(ioh.describe(test0))

with ioh.timed(f"RSF probe fit on fold {FOLDS[0]}"):
    t0 = time.time()
    probe = ioh.production_rsf(columns=train0["columns"], seed=0, **RSF_OPTIONS)
    probe.fit(train0["X"], train0["time"], train0["event"])
    cif = probe.predict_cif(test0["X"], np.asarray(T_EVAL))
    probe_seconds = time.time() - t0

n_cells = len(FOLDS) + 1          # five CV folds plus the final dev_full fit
print(f"\nR options: {probe.r_options}")
print(f"one fit + predict : {ioh.human(probe_seconds)}")
print(f"PROJECTED TOTAL   : {ioh.human(probe_seconds * n_cells)} "
      f"for {n_cells} cells")
print(f"CIF sanity        : shape {cif.shape}, range "
      f"[{cif.min():.4f}, {cif.max():.4f}], F1+F2 max {cif.sum(-1).max():.4f}")

if probe_seconds > 30 * 60:
    print("\n!! over 30 minutes per fit - consider the ntree=300 fallback "
          "above, after checking the OOB curve has plateaued.")

dev_fold0_train m1 [onehot]  37,821 rows x 104 cols   cause1 3,525  cause2 4,431  censored 29,865
dev_fold0_test m1 [onehot]  9,469 rows x 104 cols   cause1 913  cause2 1,073  censored 7,483
[2026-08-21 11:51:59] START  RSF probe fit on fold 0
[2026-08-21 11:53:48] DONE   RSF probe fit on fold 0  elapsed=1.8m

R options: {'ntree': 500, 'nodesize': 30, 'ntime': 100, 'nsplit': 10, 'perf.type': 'none', 'block.size': None, 'rf.cores': 'detectCores()'}
one fit + predict : 1.8m
PROJECTED TOTAL   : 10.9m for 6 cells
CIF sanity        : shape (9469, 3, 2), range [0.0015, 0.4128], F1+F2 max 0.5871


## 2. Build the cells

Five cross-validation cells plus one final fit on the full development window.

**The final cell evaluates on `dev_full` only.** Adding `temporal` or
`exploratory` to `evaluate_on` here would spend the project's last untouched
data on an intermediate result. They are scored once, in a separate final
notebook, after every modelling decision is locked - including this one.

In [3]:
cells, loaded = [], {}

for fold in FOLDS:
    tr, te = ioh.load_fold(FEATURES_DIR, fold, IMPUTATION, "onehot")
    loaded[fold] = (tr, te)
    cells.append(BenchmarkCell(
        model_name="rsf", fold=fold, imputation=IMPUTATION,
        train=ioh.as_cell_data(tr),
        evaluate_on={"test": ioh.as_cell_data(te)},
        factory=ioh.rsf_factory(columns=tr["columns"], **RSF_OPTIONS)))
    ioh.log(f"fold {fold}", train=f"{tr['n_rows']:,}", test=f"{te['n_rows']:,}")

# Final fit on the whole development window, scored in-sample only.
# evaluate_on deliberately contains NOTHING but dev_full - see the note above.
#
# fold = -1, not "dev_full": run_benchmark writes `fold` as a parquet column,
# and mixing ints with a string makes it an object column that pyarrow refuses
# ("Could not convert 'dev_full' with type str: tried to convert to int64").
# It would also break any later join against the other models' results, whose
# folds are integers. -1 means "fitted on the whole development window, not a
# CV fold"; the eval_set column carries the readable name.
full = ioh.load_set(FEATURES_DIR, "dev_full", "dev_full", IMPUTATION, "onehot")
DEV_FULL_FOLD = -1
cells.append(BenchmarkCell(
    model_name="rsf", fold=DEV_FULL_FOLD, imputation=IMPUTATION,
    train=ioh.as_cell_data(full),
    evaluate_on={"dev_full": ioh.as_cell_data(full)},
    factory=ioh.rsf_factory(columns=full["columns"], **RSF_OPTIONS)))

for c in cells:
    assert set(c.evaluate_on) <= {"test", "dev_full"}, \
        f"unexpected evaluation set in cell {c.fold}: {list(c.evaluate_on)}"
ioh.assert_not_held_out(*[k for c in cells for k in c.evaluate_on])

assert all(isinstance(c.fold, (int, np.integer)) for c in cells), \
    "fold must stay integer-typed or the results parquet cannot be written"
print(f"\n{len(cells)} cells: {[c.fold for c in cells]} "
      f"(-1 = full development window)")

[2026-08-21 11:53:48] fold 0  train=37,821  test=9,469
[2026-08-21 11:53:48] fold 1  train=37,823  test=9,467
[2026-08-21 11:53:48] fold 2  train=37,839  test=9,451
[2026-08-21 11:53:48] fold 3  train=37,837  test=9,453
[2026-08-21 11:53:48] fold 4  train=37,840  test=9,450

6 cells: [0, 1, 2, 3, 4, -1] (-1 = full development window)


## 3. Run

`resume=True`, so an interrupted run continues from the parquet rather than
starting over. Seeds come from `derive_seed(model, fold, imputation)`, which is
a hash rather than a counter - a resumed run therefore reproduces exactly what
an uninterrupted one would have produced, and restarting is free of
consequences.

Expect this cell to take hours. The timestamps are how a stalled run is
distinguished from a slow one.

In [4]:
RESULTS_PATH = RESULTS_DIR / "rsf_results.parquet"

with ioh.timed(f"RSF benchmark -> {RESULTS_PATH.name}"):
    results = run_benchmark(cells, np.asarray(T_EVAL), RESULTS_PATH,
                            resume=True, verbose=True)

ioh.log("written", path=RESULTS_PATH, rows=len(results))
results.head()

[2026-08-21 11:53:48] START  RSF benchmark -> rsf_results.parquet
  rsf          fold=0 m=1 test         fit    0.0s predict 121.7s coherence 4.64e-02
  rsf          fold=1 m=1 test         fit    0.0s predict 120.2s coherence 3.60e-02
  rsf          fold=2 m=1 test         fit    0.0s predict 118.3s coherence 3.54e-02
  rsf          fold=3 m=1 test         fit    0.0s predict 118.3s coherence 4.38e-02
  rsf          fold=4 m=1 test         fit    0.0s predict 118.3s coherence 3.36e-02
  rsf          fold=-1 m=1 dev_full     fit    0.0s predict 174.2s coherence 6.21e-02
[2026-08-21 12:17:50] DONE   RSF benchmark -> rsf_results.parquet  elapsed=24.0m
[2026-08-21 12:17:50] written  path=C:\Users\olf3ph\Documents\RESULTS\rsf_results.parquet  rows=36


,model,fold,imputation,eval_set,cause,horizon,auc,cindex,brier,ici,...,coherence_max,n_rows,n_events_cause,fit_seconds,predict_seconds,calibration_method,seed,converged,null_brier,beats_null
0,rsf,0,1,test,1,1.0,0.645981,0.643455,0.027521,0.001702,...,0.046439,9469,913,0.0,121.670591,fine_gray_rcs,1668261607,True,0.027762,True
1,rsf,0,1,test,1,3.0,0.645932,0.640996,0.060305,0.005116,...,0.046439,9469,913,0.0,121.670591,fine_gray_rcs,1668261607,True,0.061314,True
2,rsf,0,1,test,1,5.0,0.624598,0.624028,0.099494,0.010551,...,0.046439,9469,913,0.0,121.670591,fine_gray_rcs,1668261607,True,0.101508,True
3,rsf,0,1,test,2,1.0,0.714614,0.711234,0.026331,0.002427,...,0.046439,9469,1073,0.0,121.670591,fine_gray_rcs,1668261607,True,0.026777,True
4,rsf,0,1,test,2,3.0,0.704684,0.697453,0.059733,0.005771,...,0.046439,9469,1073,0.0,121.670591,fine_gray_rcs,1668261607,True,0.061757,True


## 4. Post-run checks

Read these before trusting anything downstream.

On the coherence column: RSF has no joint survival function of its own -
`rfsrc` returns none for the competing-risks family - so `predict_survival`
reconstructs one from the forest's own cumulative hazards by the product-limit
recursion. The residual is therefore a real property of the bagged estimator
(averaging CIFs and averaging cumulative hazards do not commute), and it runs
around 8e-02. It is **not** comparable to the Royston-Parmar residual, which is
quadrature error in a check that could have failed.

In [5]:
print("COHERENCE  max |F1 + F2 + S - 1|")
print(coherence_table(results).to_string())

print("\nCELLS NEEDING ATTENTION (non-convergence, or no better than the null)")
bad = ioh.failure_report(results)
print("  none" if bad.empty else bad.to_string(index=False))

print("\nPER-FOLD TIMING   (fold -1 = full development window)")
timing = (results.groupby(["fold", "eval_set"])[["fit_seconds",
                                                 "predict_seconds"]]
          .first().round(1))
print(timing.to_string())
print(f"\ntotal fit time: "
      f"{ioh.human(results.groupby('fold')['fit_seconds'].first().sum())}")

print("\nHELD-OUT BRIER by cause and horizon")
print(results[results.eval_set == "test"]
      .pivot_table(index="horizon", columns="cause", values="brier",
                   aggfunc="mean").round(5).to_string())

COHERENCE  max |F1 + F2 + S - 1|
          worst   average  n_records
model                               
rsf    0.062105  0.042881         36

CELLS NEEDING ATTENTION (non-convergence, or no better than the null)
  none

PER-FOLD TIMING   (fold -1 = full development window)
               fit_seconds  predict_seconds
fold eval_set                              
-1   dev_full          0.0            174.2
 0   test              0.0            121.7
 1   test              0.0            120.2
 2   test              0.0            118.3
 3   test              0.0            118.3
 4   test              0.0            118.3

total fit time: 0s

HELD-OUT BRIER by cause and horizon
cause          1        2
horizon                  
1.0      0.02596  0.02710
3.0      0.05805  0.06269
5.0      0.09662  0.10974


## 5. Summary

**Produced:** `rsf_results.parquet` in `RESULTS_DIR` - long format, one row per
(model, fold, imputation, eval_set, cause, horizon), with AUC, C-index, Brier,
Austin calibration (ICI/E50/E90/Emax), calibration-in-the-large, the coherence
residual, timings, seeds, and a per-cause `converged` flag.

**Covers:** RSF only, five development folds plus a full-development fit, at
imputation m = 1. Temporal and exploratory untouched.

**Consumed by:** the model-comparison stage, which pools this with the other
classical baselines' results and the neural variants'. Concatenate on the same
schema - `RECORD_COLUMNS` in `benchmark/base.py` - and group by model.

**To report in the methods:** the RSF settings printed by the probe cell
(`probe.r_options`), and the single-imputation deviation with the missingness
rates that justify it.

In [6]:
print(f"rsf_results.parquet : {RESULTS_PATH}")
print(f"rows                : {len(results):,}")
print(f"models              : {sorted(results.model.unique())}")
print(f"folds               : {sorted(map(str, results.fold.unique()))}")
print(f"imputations         : {sorted(results.imputation.unique())}")
print(f"eval sets           : {sorted(results.eval_set.unique())}")
print(f"RSF settings        : {RSF_OPTIONS}")

rsf_results.parquet : C:\Users\olf3ph\Documents\RESULTS\rsf_results.parquet
rows                : 36
models              : ['rsf']
folds               : ['-1', '0', '1', '2', '3', '4']
imputations         : [np.int64(1)]
eval sets           : ['dev_full', 'test']
RSF settings        : {'ntree': 500, 'nodesize': 30, 'ntime': 100, 'nsplit': 10, 'perf_type': 'none', 'block_size': None, 'rf_cores': None}
